# Module 4 demo: persistent memory with Mem0 (self-study notebook)

Instructor and student-facing self-study companion to `memory_demo.py`. Not linked from the student-facing site, but safe to hand a team that wants to read through the mechanism at their own pace.

Used alongside `lectures/dsca-module-04.html`: slide 5 (Mem0's extract-and-update mechanism) in the Lecture section, and slides 14, 15, 16, 17, and 20 in the Hands-on lab, the same slides `memory_demo.py`'s own docstring cross-references.

**Why this notebook exists alongside a plain script.** The live classroom demo is `memory_demo.py`, run twice from a terminal, because that is the most convincing way to show memory surviving a real process restart: two completely separate `python` invocations. A notebook's single, long-running kernel cannot repeat that trick by just calling a function twice, doing so would only prove an object was rebuilt, not that a process died and a new one still remembered anything. So the persistence cell below does not fake it: it launches two real, independent `python` subprocesses, exactly the same proof the live demo gives, still from inside one notebook. Every other part below is a narrated walkthrough of the same functions `memory_demo.py` defines, imported directly rather than re-implemented, so there is exactly one copy of this logic to keep correct.

**Run this once, with a real `GOOGLE_API_KEY` in `demos/.env`, then save the notebook without clearing output.** A student opening it later sees exactly what happened, without needing their own key or a live connection, the same convention `module-01/hook_demo.ipynb` uses.

**First time running this notebook?** Do the one-time setup below before anything else.

## One-time setup

Do this once per machine, before running any cell below, the same setup `../README.md` and `module-04/README.md` describe.

1. From the repository root, run `cd demos`, create or activate the shared environment (`python3 -m venv .venv`, `source .venv/bin/activate`). Select `demos/.venv` as this notebook's kernel.
2. Run the cell immediately below once, in that environment.
3. Confirm `GOOGLE_API_KEY` is set in `demos/.env` (see `demos/README.md` for exactly how to get one, free, no card).

No extra credential beyond that: the vector store is Qdrant running in local, on-disk mode, no server, no separate account.

In [13]:
from pathlib import Path

repo_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "demos" / "requirements.txt").is_file()
)
requirements_path = repo_root / "demos" / "requirements.txt"

import subprocess
import sys

print("Installing the shared demo requirements if needed...")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements_path)],
    stdout=subprocess.DEVNULL,
)
print("Requirements are ready.")


Installing the shared demo requirements if needed...
Requirements are ready.


## Setup for this notebook

Imports `memory_demo.py` as a module rather than copying its functions in, so this notebook can never quietly drift out of sync with the script the live demo actually runs. Also clears the on-disk store once, the same fresh-start `memory_demo.py`'s own `__main__` block does, so this notebook's output is reproducible run to run.

In [14]:
import shutil
import subprocess
import sys
import importlib
from pathlib import Path

module_dir = repo_root / "demos" / "module-04"
sys.path.insert(0, str(module_dir))
import memory_demo
d = importlib.reload(memory_demo)

shutil.rmtree(d.STORE_PATH, ignore_errors=True)
print(f"Store cleared at {d.STORE_PATH}")

Store cleared at /Users/ahb/Library/Mobile Documents/com~apple~CloudDocs/Documents/Teaching/LebUniv/Data Science for Conversational AI/course_data_science_for_conversational_ai/demos/module-04/mem0_store


## Part 1: extraction and conflict resolution, within one session

A first turn states a fact. A later turn changes it. Current Mem0 2.x extraction is additive, so the second `memory.add()` returns `ADD`. That exposes the stale-fact risk rather than hiding it: the notebook then uses Mem0's real `update()` to make Paris authoritative and `delete()` to remove the superseded duplicate. The four-operation lifecycle is an application policy around extraction, not magic inside one library call.

In [15]:
m1 = d.new_memory()
try:
    d.session_one(m1)
finally:
    # Embedded Qdrant permits one process per on-disk store. Release
    # this kernel's lock before Part 2 starts its independent process.
    d.close_memory(m1)

LLM extraction failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


Gemini is temporarily busy while saving the first memory; retrying in 1 second (1/2).
--- session 1, turn 1: memory.add result ---
{
  "results": [
    {
      "id": "72764ed0-27ea-4935-b489-9f553a1930ed",
      "memory": "User is based in Beirut as of September 2026",
      "event": "ADD"
    }
  ]
}

--- session 1, turn 2: memory.add result ---
{
  "results": [
    {
      "id": "21d82101-bdc3-41d7-bdb1-657a6f0859f5",
      "memory": "User relocated from Beirut to Paris for work around September 2026",
      "event": "ADD"
    }
  ]
}

--- session 1, conflict resolution: explicit UPDATE plus DELETE ---
{
  "event": "UPDATE",
  "updated_memory_id": "72764ed0-27ea-4935-b489-9f553a1930ed",
  "update_result": {
    "message": "Memory updated successfully!"
  },
  "deleted_duplicate_id": "21d82101-bdc3-41d7-bdb1-657a6f0859f5",
  "delete_result": {
    "message": "Memory deleted successfully!"
  }
}

Confirmed: the active location is Paris; the superseded duplicate was removed.

--- sessio

## Part 2: a second session, proven with a real process restart

This is the one part that a single notebook kernel cannot demonstrate honestly by just calling a function again: doing so would only prove a new Python object was built in the same process, not that memory survived past one. So this cell does not call `session_two_recall_and_cite` in this kernel. It launches a completely separate, freshly started `python` subprocess, no shared interpreter, no shared memory with this notebook at all, and has that independent process search for the fact Part 1 just wrote. This notebook's own kernel stands in for the first session; the subprocess stands in for a genuinely new one, the same proof `memory_demo.py` gives when run twice from a terminal.

In [16]:
proc = subprocess.run(
    [sys.executable, "-c", "import memory_demo as d; m = d.new_memory(); d.session_two_recall_and_cite(m)"],
    cwd=module_dir, capture_output=True, text=True,
)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)

assert "Paris" in proc.stdout, "the independent process failed to recall a fact this notebook's own kernel wrote"
print("Confirmed: a completely separate, freshly launched process recalled a fact it never itself wrote.")

--- session 2: memory.search result ---
{
  "results": [
    {
      "id": "72764ed0-27ea-4935-b489-9f553a1930ed",
      "memory": "User currently lives in Paris for work.",
      "hash": "7d2bb6ef83a6cc442fb8577879589851",
      "metadata": null,
      "score": 0.8343308653322639,
      "created_at": "2026-09-17T00:50:19.525163+00:00",
      "updated_at": "2026-09-17T00:50:23.987607+00:00",
      "user_id": "demo-user-482",
      "attributed_to": "user"
    }
  ]
}

--- session 2: answer with citation, same shape as Module 3 ---
{
  "answer": "Memory says: User currently lives in Paris for work.",
  "cited_sources": [
    "memory#72764ed0-27ea-4935-b489-9f553a1930ed"
  ]
}


Confirmed: a completely separate, freshly launched process recalled a fact it never itself wrote.


Closing the codebase, or even the whole terminal, does not lose the fact: it lives on disk at `d.STORE_PATH`, not inside any one Python process. That is the whole point of persistent memory versus session memory.

## Part 3: compaction

A fake 25-turn conversation gets folded down to a 20-turn budget: the oldest turns beyond the budget become one summary message instead of being silently dropped or left to overflow. The summarizer here is a stub, not a real Gemini call, so this part can be checked without spending a request; a real deployment would call the same model the rest of the pipeline uses. This is the same problem the lecture's summarization slide poses: a long enough conversation would eventually exceed the model's context window if every turn stayed live forever.

In [17]:
fake_long_conversation = [
    {"role": "user", "content": f"turn {i}: some detail about the ongoing project"}
    for i in range(25)
]
compacted = d.compact(fake_long_conversation, max_live_turns=20, summarize=d._naive_summary)
print(f"input turns: {len(fake_long_conversation)}")
print(f"output messages: {len(compacted)}")
print(f"first message (the folded summary): {compacted[0]}")

input turns: 25
output messages: 20
first message (the folded summary): {'role': 'system', 'content': 'Earlier conversation, summarised: covered 6 earlier turns, including: turn 0: some detail about the ongoing pr, turn 1: some detail about the ongoing pr, turn 2: some detail about the ongoing pr, turn 3: some detail about the ongoing pr, turn 4: some detail about the ongoing pr, turn 5: some detail about the ongoing pr'}


## Part 4: a real "forget me"

Calls `delete_all` for the demo user, then shows both the remaining search-hit count and stored-memory count. Both must be zero; an error-free delete call alone is not evidence. This is the concrete answer to "what happens when a user asks the agent to forget something," the Module 4 wrap discussion prompt.

In [18]:
# Reload so this cell remains correct after editing memory_demo.py in an active kernel.
d = importlib.reload(d)
m2 = d.new_memory()
forget_check = d.forget_user(m2, d.USER_ID)
d.show("forget me: deletion verification", forget_check)
assert forget_check["confirmed_deleted"], "the user's memories were not fully deleted"

--- forget me: deletion verification ---
{
  "search_hits_after_delete": 0,
  "stored_memories_after_delete": 0,
  "confirmed_deleted": true
}



## Swap in your own facts

`session_one()` and `session_two_recall_and_cite()` in `memory_demo.py` are the whole scenario: replace the two conversational turns with your own team's domain (an order-status agent remembering a customer's shipping address, a tutoring agent remembering a student's current course), and the ADD / UPDATE / search / citation flow above is unchanged, since this notebook calls those same functions rather than its own copy.

## The one API asymmetry worth remembering

`memory.add()` and `memory.delete_all()` both take `user_id` as a plain keyword argument. `memory.search()` and `memory.get_all()` both require it inside `filters={"user_id": ...}` instead, and raise `ValueError` if passed at the top level. Every call above already gets this right; check which family a new call belongs to before assuming the same calling convention carries over.